# Partie 2 : étude principale pass@k, session 10 : H4, entraînement supervisé LoRA sur les bonnes traces

Plan gelé : `docs/partie2/plan_analyse.md` (v1.0). Amendement du 23 novembre (H4) : deux adaptateurs LoRA de Qwen2.5-3B-Instruct, un par GPU, sur les trajectoires réussies du vivier (`scripts/sft_donnees.py`) : celles de l'agent PPO (session 8) et celles du modèle de départ (session 3), même nombre d'exemples. Réglages figés dans le plan. Sorties : `lora_sft_ppo/`, `lora_sft_depart/` (adaptateurs seulement).

Notebook généré par `scripts/genere_notebooks_partie2.py` : le modifier là, pas ici.


In [ ]:
BRANCH = "partie2-passk"
N_QUESTIONS, N_SAMPLES, T, TEMP = 500, 32, 4, 0.7   # valeurs par défaut ; les options d'un run priment
MAX_TOKENS, MAX_MODEL_LEN = 1024, 8192   # figés au pilote v2 ; arrêt à </search> et </answer>
FILES = [
    ['ppo'],
    ['depart'],
]


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
from pathlib import Path
REPO = "vqlentinn/memoire-rl-search"
CLONE = Path("/tmp/memoire-rl-search")
if not CLONE.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(CLONE)], check=True)
print(subprocess.run(["git", "-C", str(CLONE), "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q vllm==0.33.0 pandas pyarrow huggingface_hub peft==0.21.2 2>&1 | tail -3
!pip freeze > /kaggle/working/pip_freeze.txt
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| gpus', torch.cuda.device_count())"
import sys
sys.path.insert(0, str(CLONE))
from data.hotpot_distractor import telecharger
PARQUET = telecharger("/tmp/hotpot")
print(PARQUET)

In [ ]:
import os, threading, time
WORK = Path("/kaggle/working")
(WORK / "logs").mkdir(exist_ok=True)
status, verrou = [], threading.Lock()
# Jeux d'entraînement depuis les sorties des sessions 3 et 8 (montées sous /kaggle/input)
r = subprocess.run(["python", "-u", "scripts/sft_donnees.py", "--indir", "/kaggle/input", "--out", str(WORK / "sft")],
                   cwd=CLONE, capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
assert r.returncode == 0, "construction des données en échec"

def file_sft(gpu, sources):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    for s in sources:
        nom = f"sft_{s}"
        cmd = ["python", "-u", "scripts/train_sft.py", "--data", str(WORK / "sft" / f"sft_{s}.jsonl"),
               "--out", str(WORK / f"lora_sft_{s}")]
        t0 = time.time()
        with open(WORK / "logs" / f"{nom}.log", "a") as f:
            code = subprocess.run(cmd, cwd=CLONE, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
        with verrou:
            status.append((nom, gpu, code, round(time.time() - t0)))
            print(f"GPU {gpu} : {nom} code {code}, {time.time() - t0:.0f} s", flush=True)

fils = [threading.Thread(target=file_sft, args=(g, s)) for g, s in enumerate(FILES)]
for f in fils: f.start()
for f in fils: f.join()


In [ ]:
import json
resume = {}
for s in ("ppo", "depart"):
    m = WORK / f"lora_sft_{s}" / "meta_entrainement.json"
    resume[s] = json.loads(m.read_text()) if m.exists() else None
(WORK / "resume_s10.json").write_text(json.dumps({"status": status, "resume": resume}, indent=1, ensure_ascii=False))
print(json.dumps({"status": status, "resume": resume}, indent=1, ensure_ascii=False))
!tail -n 5 /kaggle/working/logs/*.log
echecs = [s for s in status if s[2] != 0]
if echecs or None in resume.values():
    raise RuntimeError(f"Entraînements en échec : {echecs}")
